# Lab 1 — Rail Dataset Analysis  🚆
### AI & Data Science with GenAI (Rail) · Your first data-analysis lab

**Welcome!** In the last few sessions you learned to inspect a **pandas** DataFrame
(`.head()`, `.info()`, `.describe()`), count values (`.value_counts()`), find **outliers
with the IQR method**, and draw **histograms, boxplots, scatter plots and a correlation
heatmap**.

Today you use those **same steps** on a **real railway dataset**: the official monthly
**punctuality record of French high-speed (TGV) trains**. Every row is **one route in one
month** — how many trains were scheduled, how many were cancelled, and how late they ran.

We go **one small step at a time**. Read the comments, run each cell (Shift+Enter), and do
the **🔧 Your turn** boxes. Only the very last step introduces something new.

## Step 0 — Get our tools ready and load the data
Real data is rarely tidy. This file uses a **semicolon** `;` as the separator and has
**French column names**, so we load it carefully and rename the columns to plain English.

In [ ]:
# ============================================================
# STEP 0 : import our tools and load the dataset
# ============================================================
import pandas as pd          # tables (like Excel) in Python
import numpy as np           # numbers and math
import matplotlib.pyplot as plt   # charts
import seaborn as sns        # prettier charts
%matplotlib inline

# This real file is separated by ';' (not ',') and saved with a French text encoding.
raw = pd.read_csv("sncf_tgv_regularity.csv", sep=";", encoding="utf-8-sig")

# The real column names are in French. We keep the columns we need and rename them
# so the rest of the lab is easy to read. (The DATA is untouched — only the labels change.)
columns_we_want = {
    "date":                    "month",
    "service":                 "service",          # National or International
    "gare_depart":             "from_station",
    "gare_arrivee":            "to_station",
    "duree_moyenne":           "journey_time_min",
    "nb_train_prevu":          "scheduled_trains",
    "nb_annulation":           "cancellations",
    "nb_train_retard_arrivee": "trains_late",
    "retard_moyen_arrivee":    "avg_delay_min",     # avg delay of the late arriving trains
    "nb_train_retard_sup_15":  "trains_over_15min_late",
}
df = raw[list(columns_we_want)].rename(columns=columns_we_want)

# The golden rule: always LOOK at the first few rows first.
df.head()

## Step 1 — First look: how big is it, and what's inside?
Exactly what you did with the machining data: `shape`, `info`, `describe`.

In [ ]:
# How many rows (route-months) and columns?
print("Rows and columns:", df.shape)

# What is each column, and are any values missing?
df.info()

In [ ]:
# describe() gives the summary statistics of every NUMBER column at once:
# count, mean, standard deviation, min, the quartiles (25/50/75%), and max.
df.describe()

> **Reading it:** one row is one **route in one month** (e.g. Paris → Bordeaux, Jan 2018).
> `scheduled_trains` is how many were planned, `cancellations` how many were called off,
> `avg_delay_min` the average lateness **of the trains that arrived late** (not all trains).
> In a few months this value is slightly **negative** — those late trains actually made up
> a little time on the way. That's real, not an error.

In [ ]:
# 🔧 Your turn:
# Print how many ROWS the data has, using df.shape (the first number).
# Hint: df.shape[0]

## Step 2 — Counting categories
`nunique()` counts how many *different* values there are.
`value_counts()` counts how many rows fall into each category.

In [ ]:
# How many different departure stations, and how many of each service type?
print("Number of departure stations :", df["from_station"].nunique())
print()
print(df["service"].value_counts())

In [ ]:
# Which departure stations appear most often? (busiest in the records)
top_stations = df["from_station"].value_counts().head(10)

plt.figure(figsize=(9, 4))
plt.bar(top_stations.index, top_stations.values, color="skyblue", edgecolor="black")
plt.title("Top 10 busiest departure stations (by number of records)")
plt.xlabel("Departure station")
plt.ylabel("Number of route-months")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

> **Insight:** the big Paris terminals dominate — they are the hubs of the TGV network,
> so they appear on the most routes.

In [ ]:
# 🔧 Your turn:
# Show the 10 busiest ARRIVAL stations ("to_station") the same way.
# Hint: copy the bar-chart cell and change the column.

## Step 3 — The shape of one number (a histogram)
A histogram shows how a single number is spread out. Let's look at the **average delay**.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["avg_delay_min"], bins=40, color="skyblue", edgecolor="black")
plt.title("Distribution of average arrival delay (of late trains)")
plt.xlabel("Average delay (minutes)")
plt.ylabel("Number of route-months")
plt.tight_layout()
plt.show()

print(f"Average of the column : {df['avg_delay_min'].mean():.1f} min")
print(f"Worst value           : {df['avg_delay_min'].max():.1f} min")
print(f"Skewness              : {df['avg_delay_min'].skew():.2f}")

> **Insight:** most route-months cluster around a typical delay, with a long tail to the
> right — a **right-skewed** distribution (positive skewness). You met skewness in the
> statistics session.

In [ ]:
# 🔧 Your turn:
# Draw a histogram of "journey_time_min" (average journey time).
# Copy the cell above and change the column and the labels.

## Step 4 — Finding outliers with the IQR method
The **same recipe** you used on the machining data. Let's find months where a route had an
**unusually high number of cancellations**.

In [ ]:
# ============================================================
# STEP 4 : IQR outlier detection on cancellations
# ============================================================
col = "cancellations"

Q1  = df[col].quantile(0.25)     # 25th percentile
Q3  = df[col].quantile(0.75)     # 75th percentile
IQR = Q3 - Q1                    # the middle 50% spread

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1            : {Q1:.1f}")
print(f"Q3            : {Q3:.1f}")
print(f"IQR           : {IQR:.1f}")
print(f"Upper bound   : {upper_bound:.1f}  (more cancellations than this = outlier)")

In [ ]:
# A boxplot shows the same idea as a picture. Dots past the right whisker = outliers.
plt.figure(figsize=(8, 3))
plt.boxplot(df[col], vert=False, patch_artist=True,
            boxprops=dict(facecolor="lightgreen"))
plt.title("Cancellations per route-month — boxplot")
plt.xlabel("Cancellations")
plt.tight_layout()
plt.show()

In [ ]:
# WHICH route-months are the outliers? Let's see the worst ones.
outliers = df[df[col] > upper_bound].sort_values(col, ascending=False)
print("Number of outlier route-months:", len(outliers))
outliers[["month", "from_station", "to_station", "cancellations", "scheduled_trains"]].head(10)

> **Insight:** the biggest cancellation spikes line up with **known strike periods** on the
> French railway (for example spring 2018 and late 2019). An outlier here is a real event,
> not bad data — the sort of context a railway expert brings to the numbers.

In [ ]:
# 🔧 Your turn:
# Run the SAME IQR steps for the column "avg_delay_min".
# How many route-months have an unusually high average delay?
# Hint: copy the Q1/Q3/IQR block AND the outlier-count line
#       (df[df[col] > upper_bound]), changing col to "avg_delay_min".
#       Tip: use fresh names (col2, Q1b, upper2) so you don't overwrite
#       the values from the cancellations cells above.

## Step 5 — Do two numbers move together? (a scatter plot)
Do **busier** routes (more scheduled trains) also have **more late** trains?

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["scheduled_trains"], df["trains_late"], alpha=0.3, color="teal")
plt.title("Do busier routes have more late trains?")
plt.xlabel("Scheduled trains")
plt.ylabel("Trains late on arrival")
plt.tight_layout()
plt.show()

r = df["scheduled_trains"].corr(df["trains_late"])
print(f"Correlation between scheduled trains and late trains: {r:.2f}")

> **Insight:** a clear upward trend — the more trains a route runs, the more end up late
> (simply because there are more of them). A solid positive correlation.

In [ ]:
# 🔧 Your turn:
# Make a scatter plot of "journey_time_min" (x) vs "avg_delay_min" (y),
# then print their correlation with .corr().
# Hint: copy the scatter cell above; change the two columns and the labels.

## Step 6 — All the relationships at once (correlation heatmap)
The **same `sns.heatmap`** you used on the machining data, across every number column.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation between the number columns")
plt.tight_layout()
plt.show()

> **Insight:** `trains_late` and `trains_over_15min_late` move almost perfectly together
> (dark red). `scheduled_trains` is strongly linked to both. But `cancellations` is near
> **0** with everything — cancellations are driven by one-off events (strikes), not by how
> busy or delayed a route normally is. Strong vs weak relationships, side by side.

In [ ]:
# 🔧 Your turn:
# The heatmap suggested cancellations are unrelated to the rest.
# Check it directly: print the correlation between "cancellations" and "scheduled_trains".
# Is it strong (near +1 / -1) or weak (near 0)?
# Hint: df["cancellations"].corr(df["scheduled_trains"])

## Step 7 — One new tool: `groupby` (grouping and summarising) ⭐
Everything so far you had already seen. Here is **one new idea**, and it is a big one.

`groupby` splits the data into groups (say, by departure station) and calculates something
for **each group** — like the average delay per station. It answers *"which station's trains
run latest on average?"*

In [ ]:
# groupby("from_station") -> one group per departure station
# ["avg_delay_min"].mean() -> the average delay inside each group
worst = (df.groupby("from_station")["avg_delay_min"]
           .mean()
           .sort_values(ascending=False)
           .head(10))
print(worst.round(1))

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(worst.index, worst.values, color="lightgreen", edgecolor="black")
plt.title("Top 10 departure stations by average arrival delay")
plt.xlabel("Departure station")
plt.ylabel("Average delay (minutes)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

> **Insight:** the worst-delayed departures are the long southern and cross-border routes
> (Toulouse, Nice, Perpignan, Geneva…) — long journeys have more chances to accumulate
> delay. `groupby` turned 12,000 rows into one clear ranking; it is the workhorse you will
> use again and again.

In [ ]:
# 🔧 Your turn (practise the new tool!):
# Use groupby to find the AVERAGE number of cancellations for each service
# (National vs International). Which service is cancelled more on average?
# Hint: df.groupby("service")["cancellations"].mean()

## 🏁 Challenge (put it all together)
Now **you** lead. **Do both** of these, then write a one-line finding under each:

1. Which **route** (from_station → to_station) has the **most cancellations** in total?
   *(Hint: groupby the two station columns, `.sum()` the cancellations, sort, take the top.)*
2. Make a **boxplot** of `avg_delay_min` for the **5 busiest departure stations**.
   *(Hint: `top5 = df["from_station"].value_counts().head(5).index`, then
   `sns.boxplot(x="from_station", y="avg_delay_min", data=df[df["from_station"].isin(top5)])`.)*

**Stretch (optional):** does National or International run later on average?

In [ ]:
# 🏁 Challenge — your workspace (write your own code here)

# Part 1: the route with the most cancellations


# Part 2: boxplot of avg_delay_min for the 5 busiest departure stations

## ✅ Wrap up
You just ran a full **exploratory data analysis (EDA)** on a real railway dataset, using
the same steps as before plus one new tool (`groupby`):

**load → look → count → distributions → outliers → relationships → grouped summary.**

Write your **three findings** and **one recommendation** for the operations team in your
Student Workbook, then save this notebook (File > Save) and submit it.

*Data: SNCF Open Data - "Regularite mensuelle TGV" (monthly TGV punctuality). Licence
Ouverte / Open Licence (Etalab) - free to use, including commercially, with attribution.*